## PointNet++ MSG on ModelNet40 with Colab T4 Training

In [1]:
# ---- Install non-default packages ----
!pip install -q trimesh

# ---- Standard imports ----
import os
import time
import io
import glob
import urllib.request
import zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import trimesh

# ---- Reproducibility ----
SEED = 1234
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ---- Device check ----
if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU enabled : {gpu_name}  ({gpu_mem:.1f} GB)")
else:
    raise RuntimeError("No GPU detected! Runtime -> Change runtime type -> T4 GPU")

# ---- Paths (Colab working directory is /content) ----
WORK_DIR       = "/content"
DATA_DIR_M40   = os.path.join(WORK_DIR, "ModelNet40")
ZIP_PATH_M40   = os.path.join(WORK_DIR, "ModelNet40.zip")
CACHE_DIR_M40  = os.path.join(WORK_DIR, "cache_modelnet40_1024pts")
CHECKPOINT_DIR = os.path.join(WORK_DIR, "checkpoints")
os.makedirs(CACHE_DIR_M40, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"\nPyTorch version : {torch.__version__}")
print(f"CUDA version    : {torch.version.cuda}")
print(f"Device          : {device}")
print(f"Working dir     : {WORK_DIR}")
print(f"\nReady. Continue to Cell 3 (download ModelNet40).")

GPU enabled : Tesla T4  (14.6 GB)

PyTorch version : 2.11.0+cu128
CUDA version    : 12.8
Device          : cuda
Working dir     : /content

Ready. Continue to Cell 3 (download ModelNet40).


## Downloading and extracting ModelNet40

In [2]:
URLS_M40 = [
    "http://modelnet.cs.princeton.edu/ModelNet40.zip",
    "https://3dvision.princeton.edu/projects/2014/3DShapeNets/ModelNet40.zip",
]


def download_with_progress(url, dest):
    """Download with a percentage progress bar that updates in place."""
    def hook(block_num, block_size, total_size):
        downloaded = block_num * block_size
        if total_size > 0:
            percent = min(100, downloaded * 100 / total_size)
            mb_done  = downloaded / 1024**2
            mb_total = total_size / 1024**2
            print(f"\r  {percent:5.1f}%  ({mb_done:7.1f} / {mb_total:7.1f} MB)", end="")
    urllib.request.urlretrieve(url, dest, reporthook=hook)
    print()


# ---- A. Download (skip if zip exists and is at least 1 GB) ----
if os.path.exists(ZIP_PATH_M40) and os.path.getsize(ZIP_PATH_M40) > 1e9:
    size_mb = os.path.getsize(ZIP_PATH_M40) / 1024**2
    print(f"Zip already exists ({size_mb:.1f} MB) -- skipping download")
else:
    print("Downloading ModelNet40.zip (~1.7 GB)...")
    download_success = False
    for url in URLS_M40:
        try:
            print(f"  Trying: {url}")
            download_with_progress(url, ZIP_PATH_M40)
            download_success = True
            print(f"  OK")
            break
        except Exception as e:
            print(f"  FAILED: {e}")
            if os.path.exists(ZIP_PATH_M40):
                os.remove(ZIP_PATH_M40)
    if not download_success:
        raise RuntimeError("All Princeton URLs failed. Try later.")


# ---- B. Extract (skip if a known class folder already exists) ----
if os.path.isdir(os.path.join(DATA_DIR_M40, "airplane")):
    print(f"\nModelNet40 already extracted -- skipping")
else:
    print(f"\nExtracting ModelNet40.zip to {WORK_DIR}/ ...")
    with zipfile.ZipFile(ZIP_PATH_M40, "r") as zf:
        zf.extractall(WORK_DIR)
    print(f"  OK")


# ---- C. Verify class folders ----
CLASSES_M40 = sorted([
    d for d in os.listdir(DATA_DIR_M40)
    if os.path.isdir(os.path.join(DATA_DIR_M40, d))
    and not d.startswith(("_", "."))
])
CLASS_TO_IDX_M40 = {c: i for i, c in enumerate(CLASSES_M40)}
IDX_TO_CLASS_M40 = {i: c for c, i in CLASS_TO_IDX_M40.items()}

print(f"\nFound {len(CLASSES_M40)} class folders")
assert len(CLASSES_M40) == 40, f"Expected 40 classes, found {len(CLASSES_M40)}"
print("ModelNet40 ready.")
print(f"First 5 classes: {CLASSES_M40[:5]}")
print(f"Last  5 classes: {CLASSES_M40[-5:]}")

Zip already exists (1944.7 MB) -- skipping download

ModelNet40 already extracted -- skipping

Found 40 class folders
ModelNet40 ready.
First 5 classes: ['airplane', 'bathtub', 'bed', 'bench', 'bookshelf']
Last  5 classes: ['toilet', 'tv_stand', 'vase', 'wardrobe', 'xbox']


## Cached datasets and DataLoaders

In [3]:

train_cache_path = "/content/cache_modelnet40_1024pts/train.npz"
test_cache_path  = "/content/cache_modelnet40_1024pts/test.npz"


assert os.path.exists(train_cache_path), f"Missing: {train_cache_path} -- re-upload it"
assert os.path.exists(test_cache_path),  f"Missing: {test_cache_path} -- re-upload it"


class AugmentedCachedModelNet40(Dataset):
    """Cached ModelNet40 with optional z-rotation + jitter."""
    def __init__(self, cache_path, augment=False, jitter_sigma=0.02, jitter_clip=0.05):
        data = np.load(cache_path)
        self.points = data["points"]
        self.labels = data["labels"]
        self.augment = augment
        self.jitter_sigma = jitter_sigma
        self.jitter_clip = jitter_clip

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        pts = self.points[idx].copy()
        if self.augment:
            theta = np.random.uniform(0, 2 * np.pi)
            c, s = np.cos(theta), np.sin(theta)
            R = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]], dtype=np.float32)
            pts = pts @ R.T
            jitter = np.random.normal(0, self.jitter_sigma, pts.shape).astype(np.float32)
            jitter = np.clip(jitter, -self.jitter_clip, self.jitter_clip)
            pts = pts + jitter
        return torch.from_numpy(pts), int(self.labels[idx])


BATCH_SIZE = 8

train_aug_m40  = AugmentedCachedModelNet40(train_cache_path, augment=True)
test_clean_m40 = AugmentedCachedModelNet40(test_cache_path,  augment=False)

train_loader_m40 = DataLoader(
    train_aug_m40, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True, drop_last=True
)
test_loader_m40  = DataLoader(
    test_clean_m40, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

print(f"Train (augmented): {len(train_aug_m40)} samples, {len(train_loader_m40)} batches")
print(f"Test  (clean)    : {len(test_clean_m40)} samples, {len(test_loader_m40)} batches")

# Sanity batch
pts, lbls = next(iter(train_loader_m40))
print(f"\nOne batch: points {tuple(pts.shape)}, labels {tuple(lbls.shape)}")

Train (augmented): 9843 samples, 1230 batches
Test  (clean)    : 2468 samples, 309 batches

One batch: points (8, 1024, 3), labels (8,)


## FPS + Ball Query + index_points + SSG SetAbstraction

In [4]:
def farthest_point_sample(xyz, npoint):
    """Farthest Point Sampling — (B,N,3) -> (B,npoint) indices."""
    B, N, _ = xyz.shape
    device = xyz.device
    centroids = torch.zeros(B, npoint, dtype=torch.long, device=device)
    distance = torch.full((B, N), float("inf"), device=device)
    farthest = torch.randint(0, N, (B,), dtype=torch.long, device=device)
    batch_idx = torch.arange(B, dtype=torch.long, device=device)
    for i in range(npoint):
        centroids[:, i] = farthest
        centroid_xyz = xyz[batch_idx, farthest, :].unsqueeze(1)
        dist = ((xyz - centroid_xyz) ** 2).sum(dim=-1)
        distance = torch.minimum(distance, dist)
        farthest = distance.argmax(dim=-1)
    return centroids


def index_points(points, idx):
    """Gather points by indices."""
    B = points.shape[0]
    view_shape = list(idx.shape); view_shape[1:] = [1] * (len(view_shape) - 1)
    repeat_shape = list(idx.shape); repeat_shape[0] = 1
    batch_indices = (torch.arange(B, dtype=torch.long, device=points.device)
                     .view(view_shape).repeat(repeat_shape))
    return points[batch_indices, idx, :]


def ball_query(radius, nsample, xyz, new_xyz):
    """Find up to nsample neighbors within radius for each query point."""
    B, N, _ = xyz.shape
    _, S, _ = new_xyz.shape
    device = xyz.device
    group_idx = (torch.arange(N, dtype=torch.long, device=device)
                 .view(1, 1, N).repeat(B, S, 1))
    sqrdists = ((new_xyz.unsqueeze(2) - xyz.unsqueeze(1)) ** 2).sum(dim=-1)
    group_idx[sqrdists > radius ** 2] = N
    group_idx = group_idx.sort(dim=-1)[0][:, :, :nsample]
    group_first = group_idx[:, :, 0:1].repeat(1, 1, nsample)
    mask = group_idx == N
    group_idx[mask] = group_first[mask]
    return group_idx


class SetAbstraction(nn.Module):
    """Single-Scale Set Abstraction: FPS -> Ball Query -> mini-PointNet -> max-pool."""
    def __init__(self, npoint, radius, nsample, in_channel, mlp):
        super().__init__()
        self.npoint, self.radius, self.nsample = npoint, radius, nsample
        self.mlp_convs, self.mlp_bns = nn.ModuleList(), nn.ModuleList()
        last = in_channel
        for out_channel in mlp:
            self.mlp_convs.append(nn.Conv2d(last, out_channel, 1))
            self.mlp_bns.append(nn.BatchNorm2d(out_channel))
            last = out_channel

    def forward(self, xyz, features=None):
        fps_idx = farthest_point_sample(xyz, self.npoint)
        new_xyz = index_points(xyz, fps_idx)
        nn_idx = ball_query(self.radius, self.nsample, xyz, new_xyz)
        grouped_xyz = index_points(xyz, nn_idx) - new_xyz.unsqueeze(2)
        if features is not None:
            grouped_features = index_points(features, nn_idx)
            new_features = torch.cat([grouped_xyz, grouped_features], dim=-1)
        else:
            new_features = grouped_xyz
        new_features = new_features.permute(0, 3, 1, 2).contiguous()
        for conv, bn in zip(self.mlp_convs, self.mlp_bns):
            new_features = F.relu(bn(conv(new_features)))
        new_features = new_features.max(dim=-1)[0]
        return new_xyz, new_features.permute(0, 2, 1).contiguous()


class GlobalSetAbstraction(nn.Module):
    """Final SA layer: aggregate ALL points into one global descriptor."""
    def __init__(self, in_channel, mlp):
        super().__init__()
        self.mlp_convs, self.mlp_bns = nn.ModuleList(), nn.ModuleList()
        last = in_channel
        for out_channel in mlp:
            self.mlp_convs.append(nn.Conv1d(last, out_channel, 1))
            self.mlp_bns.append(nn.BatchNorm1d(out_channel))
            last = out_channel

    def forward(self, xyz, features):
        x = torch.cat([xyz, features], dim=-1).permute(0, 2, 1)
        for conv, bn in zip(self.mlp_convs, self.mlp_bns):
            x = F.relu(bn(conv(x)))
        return x.max(dim=-1)[0]


print("Building blocks defined: FPS, Ball Query, SetAbstraction, GlobalSetAbstraction")

Building blocks defined: FPS, Ball Query, SetAbstraction, GlobalSetAbstraction


## Sanity check of FPS+SA1+SA2 on real batch

In [5]:
# Quick sanity: run one batch through SSG-style SA1 + SA2 to verify everything works on CUDA
sa1_sanity = SetAbstraction(
    npoint=512, radius=0.2, nsample=32, in_channel=3, mlp=[64, 64, 128]
).to(device)
sa2_sanity = SetAbstraction(
    npoint=128, radius=0.4, nsample=64, in_channel=128+3, mlp=[128, 128, 256]
).to(device)

pts, lbls = next(iter(train_loader_m40))
pts = pts.to(device)

t0 = time.time()
new_xyz_1, new_feat_1 = sa1_sanity(pts, features=None)
new_xyz_2, new_feat_2 = sa2_sanity(new_xyz_1, new_feat_1)
elapsed_ms = (time.time() - t0) * 1000

print(f"Input  : {tuple(pts.shape)}")
print(f"SA1 out: xyz {tuple(new_xyz_1.shape)}, features {tuple(new_feat_1.shape)}")
print(f"SA2 out: xyz {tuple(new_xyz_2.shape)}, features {tuple(new_feat_2.shape)}")
print(f"Forward time on T4: {elapsed_ms:.1f} ms per batch (SSG-style)")
print(f"Expected MSG forward time: ~{elapsed_ms*2:.0f} ms (3 radii instead of 1)")
print(f"Estimated MSG epoch time: ~{elapsed_ms*2*307/1000:.0f}s = ~{elapsed_ms*2*307/60000:.1f} min")

Input  : (8, 1024, 3)
SA1 out: xyz (8, 512, 3), features (8, 512, 128)
SA2 out: xyz (8, 128, 3), features (8, 128, 256)
Forward time on T4: 688.1 ms per batch (SSG-style)
Expected MSG forward time: ~1376 ms (3 radii instead of 1)
Estimated MSG epoch time: ~422s = ~7.0 min


## MSG architecture

In [6]:
class SetAbstractionMSG(nn.Module):
    """Multi-Scale Grouping: for each centroid, gather neighbors at multiple
    radii in parallel, run a separate mini-PointNet per radius, concat features.

    This is the only architectural difference from SSG — instead of one
    (radius, nsample, mlp) per layer, MSG has 3 of them and concatenates results.
    """
    def __init__(self, npoint, radii, nsamples, in_channel, mlps):
        super().__init__()
        assert len(radii) == len(nsamples) == len(mlps), "all scale lists must match"
        self.npoint = npoint
        self.radii = radii
        self.nsamples = nsamples

        # One conv block per scale
        self.conv_blocks = nn.ModuleList()
        self.bn_blocks = nn.ModuleList()
        for mlp in mlps:
            convs = nn.ModuleList()
            bns = nn.ModuleList()
            last = in_channel + 3   # +3 for centered xyz
            for out_channel in mlp:
                convs.append(nn.Conv2d(last, out_channel, 1))
                bns.append(nn.BatchNorm2d(out_channel))
                last = out_channel
            self.conv_blocks.append(convs)
            self.bn_blocks.append(bns)

    def forward(self, xyz, features=None):
        # FPS picks centroids ONCE (shared across all scales)
        fps_idx = farthest_point_sample(xyz, self.npoint)
        new_xyz = index_points(xyz, fps_idx)

        scale_outputs = []
        for i, (radius, nsample) in enumerate(zip(self.radii, self.nsamples)):
            # Ball Query at this scale
            nn_idx = ball_query(radius, nsample, xyz, new_xyz)
            grouped_xyz = index_points(xyz, nn_idx) - new_xyz.unsqueeze(2)

            if features is not None:
                grouped_features = index_points(features, nn_idx)
                grouped = torch.cat([grouped_xyz, grouped_features], dim=-1)
            else:
                grouped = grouped_xyz

            # Mini-PointNet for this scale
            grouped = grouped.permute(0, 3, 1, 2).contiguous()
            for conv, bn in zip(self.conv_blocks[i], self.bn_blocks[i]):
                grouped = F.relu(bn(conv(grouped)))

            # Max-pool over neighbors of this scale
            pooled = grouped.max(dim=-1)[0]  # (B, mlp[-1], npoint)
            scale_outputs.append(pooled)

        # Concatenate features across scales
        new_features = torch.cat(scale_outputs, dim=1)  # (B, sum_of_outputs, npoint)
        return new_xyz, new_features.permute(0, 2, 1).contiguous()


class PointNetPlusPlusMSG(nn.Module):
    """PointNet++ MSG for ModelNet40 classification.

    Paper architecture (Table 1, MSG variant):
        SA1 MSG:    npoint=512, radii=[0.1,0.2,0.4], nsample=[16,32,128]
                    mlps=[[32,32,64], [64,64,128], [64,96,128]] -> 320 ch
        SA2 MSG:    npoint=128, radii=[0.2,0.4,0.8], nsample=[32,64,128]
                    mlps=[[64,64,128], [128,128,256], [128,128,256]] -> 640 ch
        Global SA:  mlp=[256, 512, 1024]
        Classifier: FC(1024 -> 512 -> 256 -> num_classes), Dropout(0.5)
    """
    def __init__(self, num_classes=40, dropout=0.5):
        super().__init__()
        self.sa1 = SetAbstractionMSG(
            npoint=512,
            radii=[0.1, 0.2, 0.4],
            nsamples=[16, 32, 128],
            in_channel=0,
            mlps=[[32, 32, 64], [64, 64, 128], [64, 96, 128]],
        )
        # SA1 output: 64 + 128 + 128 = 320 channels

        self.sa2 = SetAbstractionMSG(
            npoint=128,
            radii=[0.2, 0.4, 0.8],
            nsamples=[32, 64, 128],
            in_channel=320,
            mlps=[[64, 64, 128], [128, 128, 256], [128, 128, 256]],
        )
        # SA2 output: 128 + 256 + 256 = 640 channels

        self.sa_global = GlobalSetAbstraction(in_channel=640 + 3, mlp=[256, 512, 1024])

        self.classifier = nn.Sequential(
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512,  256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256,  num_classes),
        )

    def forward(self, xyz):
        l1_xyz, l1_features = self.sa1(xyz, features=None)
        l2_xyz, l2_features = self.sa2(l1_xyz, l1_features)
        global_features = self.sa_global(l2_xyz, l2_features)
        return self.classifier(global_features)


# Quick check
model = PointNetPlusPlusMSG(num_classes=40, dropout=0.5).to(device)
n_total = sum(p.numel() for p in model.parameters())
print(f"PointNet++ MSG total params: {n_total:,}")
print(f"(SSG was ~1.5M; MSG is bigger because of 3 mini-PointNets per SA)")

PointNet++ MSG total params: 1,747,368
(SSG was ~1.5M; MSG is bigger because of 3 mini-PointNets per SA)


In [7]:
print(torch.cuda.memory_allocated()/1024**3)
print(torch.cuda.memory_reserved()/1024**3)

0.5536646842956543
0.724609375


In [8]:
pts, lbls = next(iter(train_loader_m40))
pts, lbls = pts.to(device), lbls.to(device)

with torch.no_grad():
    out = model(pts)

print("Input :", pts.shape)
print("Output:", out.shape)

print("Allocated:",
      torch.cuda.memory_allocated()/1024**3,
      "GB")

print("Reserved:",
      torch.cuda.memory_reserved()/1024**3,
      "GB")

Input : torch.Size([8, 1024, 3])
Output: torch.Size([8, 40])
Allocated: 0.5625777244567871 GB
Reserved: 2.009765625 GB


In [9]:
model.train()

opt = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

opt.zero_grad()

logits = model(pts)

loss = F.cross_entropy(
    logits,
    lbls
)

loss.backward()

opt.step()

print("Loss:", loss.item())

print("Allocated:",
      torch.cuda.memory_allocated()/1024**3,
      "GB")

print("Reserved:",
      torch.cuda.memory_reserved()/1024**3,
      "GB")

Loss: 4.041665077209473
Allocated: 0.5910429954528809 GB
Reserved: 3.927734375 GB


## Training MSG with 60 epochs

In [ ]:
best_path_msg = "/content/checkpoints/pointnetpp_msg_modelnet40_best.pt"

# Fresh model
model = PointNetPlusPlusMSG(
    num_classes=40,
    dropout=0.5
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

NUM_EPOCHS = 60

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
    eta_min=1e-5
)

# New PyTorch AMP API
scaler = torch.amp.GradScaler("cuda")

history_msg = {
    "train_loss": [],
    "train_acc": [],
    "val_acc": [],
    "lr": []
}

best_val_acc_msg = 0.0

torch.cuda.empty_cache()

print(f"Training PointNet++ MSG on Colab T4 -- {NUM_EPOCHS} epochs")
print(f"Recipe: aug + cosine LR + AMP + dropout 0.5")
print(f"Our SSG (MPS): 0.8825  |  Paper SSG: 0.907  |  Paper MSG: 0.919\n")

print(
    f"{'epoch':>5}  "
    f"{'lr':>8}  "
    f"{'train_loss':>10}  "
    f"{'train_acc':>10}  "
    f"{'val_acc':>9}  "
    f"{'time':>7}"
)

print("-" * 70)

for epoch in range(1, NUM_EPOCHS + 1):

    t0 = time.time()
    current_lr = optimizer.param_groups[0]["lr"]

    # =====================
    # TRAIN
    # =====================
    model.train()

    loss_sum = 0.0
    correct = 0
    total = 0

    for pts, lbls in train_loader_m40:

        pts = pts.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda"):

            logits = model(pts)

            loss = F.cross_entropy(
                logits,
                lbls
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loss_sum += loss.item() * pts.size(0)

        correct += (
            logits.argmax(dim=1) == lbls
        ).sum().item()

        total += pts.size(0)

    scheduler.step()

    train_loss = loss_sum / total
    train_acc = correct / total

    # =====================
    # VALIDATION
    # =====================
    model.eval()

    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for pts, lbls in test_loader_m40:

            pts = pts.to(device, non_blocking=True)
            lbls = lbls.to(device, non_blocking=True)

            with torch.amp.autocast("cuda"):
                logits = model(pts)

            val_correct += (
                logits.argmax(dim=1) == lbls
            ).sum().item()

            val_total += pts.size(0)

    val_acc = val_correct / val_total

    history_msg["train_loss"].append(train_loss)
    history_msg["train_acc"].append(train_acc)
    history_msg["val_acc"].append(val_acc)
    history_msg["lr"].append(current_lr)

    flag = ""

    if val_acc > best_val_acc_msg:

        best_val_acc_msg = val_acc

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_val_acc": best_val_acc_msg,
            },
            best_path_msg
        )

        flag = "  *"

    print(
        f"{epoch:>5}  "
        f"{current_lr:>8.2e}  "
        f"{train_loss:>10.4f}  "
        f"{train_acc:>10.4f}  "
        f"{val_acc:>9.4f}  "
        f"{time.time()-t0:>6.1f}s{flag}"
    )

    if epoch % 5 == 0:
        print(
            f"      GPU Reserved: "
            f"{torch.cuda.memory_reserved()/1024**3:.2f} GB"
        )

print("\n" + "=" * 70)
print("FINAL RESULTS")
print("=" * 70)

print(f"Our SSG (MPS, 60ep)   : 0.8825")
print(f"Our MSG (T4, 60ep)    : {best_val_acc_msg:.4f}")
print(f"Paper SSG            : 0.9070")
print(f"Paper MSG            : 0.9190")
print(f"Checkpoint saved     : {best_path_msg}")

# =====================
# PLOT
# =====================

fig, ax = plt.subplots(figsize=(11, 5))

epochs = range(1, NUM_EPOCHS + 1)

ax.plot(
    epochs,
    history_msg["train_acc"],
    "-o",
    markersize=3,
    label="MSG Train",
    alpha=0.6
)

ax.plot(
    epochs,
    history_msg["val_acc"],
    "-o",
    markersize=3,
    linewidth=2,
    label="MSG Validation"
)

ax.axhline(
    0.8825,
    linestyle="--",
    alpha=0.5,
    label="Our SSG (0.883)"
)

ax.axhline(
    0.907,
    linestyle="--",
    alpha=0.6,
    label="Paper SSG (0.907)"
)

ax.axhline(
    0.919,
    linestyle="--",
    alpha=0.6,
    label="Paper MSG (0.919)"
)

ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_title("PointNet++ MSG on ModelNet40 (T4)")
ax.set_ylim(0.5, 1.0)

ax.grid(alpha=0.3)
ax.legend(loc="lower right")

plt.tight_layout()
plt.show()

Training PointNet++ MSG on Colab T4 -- 60 epochs
Recipe: aug + cosine LR + AMP + dropout 0.5
Our SSG (MPS): 0.8825  |  Paper SSG: 0.907  |  Paper MSG: 0.919

epoch        lr  train_loss   train_acc    val_acc     time
----------------------------------------------------------------------
    1  1.00e-03      1.9999      0.4574     0.6414   308.1s  *
    2  9.99e-04      1.4225      0.5856     0.6969   310.6s  *
    3  9.97e-04      1.1897      0.6566     0.7666   308.8s  *
    4  9.94e-04      1.1053      0.6740     0.7865   311.0s  *
    5  9.89e-04      0.9983      0.7036     0.7873   312.4s  *
      GPU Reserved: 2.24 GB
    6  9.83e-04      0.9142      0.7268     0.8006   311.7s  *
    7  9.76e-04      0.8772      0.7340     0.8339   311.5s  *
    8  9.67e-04      0.8306      0.7513     0.8177   310.5s
    9  9.57e-04      0.7810      0.7658     0.8298   309.0s
   10  9.46e-04      0.7670      0.7639     0.8156   308.0s
      GPU Reserved: 2.24 GB
   11  9.34e-04      0.7122      0